# Module 2 · Paper2Agent: from papers to tools

**Swarm in the Lab** · BiocAsia 2026 · 14:40–15:30

Module 1's team wrote a strategy full of analyses. Here you give the agents the tools to run them: **paper agents**, which are MCP servers that Paper2Agent built from a paper's own code and tested against the paper's own results.

| Time | Section | What you do |
|---|---|---|
| 14:40 | **1 · Walkthrough** | Tour a recorded Paper2Agent build: its stages, agents, generated code and tests |
| 14:55 | **2 · Hands-on** | Connect the Scanpy and TISSUE paper agents to Claude Code, ask for analyses in plain language, then inspect and re-run the code that actually ran |
| 15:20 | **3 · Discussion** | What makes a paper "agentifiable", and why that's also a reproducibility test |
| — | **⭐ Stretch** | Start a Paper2Agent build on your own repository, to finish after the session |

**What you leave with:** `analysis_log.json` (what ran, what came back, and whether you believe it) and `paper_agents.json` (how to connect the paper agents), both in your outputs folder. In Module 3 the Virtual Biotech team gets both.

**Falling behind?** Set `USE_CACHE = True` in the next cell and re-run. The tool list and analysis results then load from pre-run copies.

## 0 · Setup

In [ ]:
from pathlib import Path

WORKSHOP_DIR   = Path.home() / "workshop"                                 # same folder as Module 1
OUTPUT_DIR     = WORKSHOP_DIR / "outputs" / "module2"
STRATEGY_PATH  = WORKSHOP_DIR / "outputs" / "module1" / "research_strategy.json"
CACHE_DIR      = WORKSHOP_DIR / "cache" / "module2"
TARGET         = "CD276"                                                  # gene symbol as it appears in the dataset

DATASET_PATH   = WORKSHOP_DIR / "data" / "workshop.h5ad"                  # 20,000 lung adenocarcinoma cells (LuCA atlas), raw counts
TISSUE_DATA    = WORKSHOP_DIR / "data" / "tissue"                         # TISSUE's own example data (mouse brain, 33 genes)
SPATIAL_DATA   = (f"spatial counts {TISSUE_DATA / 'Spatial_count.txt'}, spot locations {TISSUE_DATA / 'Locations.txt'} "
                  f"and matching scRNA-seq counts {TISSUE_DATA / 'scRNA_count.txt'} (TISSUE's example data: mouse brain, 33 genes)")
RECORDED_BUILD = WORKSHOP_DIR / "paper2agent_builds" / "Scanpy_Agent"     # [PLACEHOLDER] a finished Paper2Agent project
PAPER2AGENT_DIR = WORKSHOP_DIR / "Paper2Agent"                            # the cloned Paper2Agent repository

# Pre-built paper agents on this machine (local copies of the Paper2Agent team's hosted servers, run over stdio).
# The tools save every file they make in their output folder; they have no per-call folder setting.
PAPER_AGENT_OUTPUTS = {"scanpy": OUTPUT_DIR / "paper_agent_files" / "scanpy",
                       "tissue": OUTPUT_DIR / "paper_agent_files" / "tissue"}
PAPER_AGENTS = {
    "scanpy": {
        "type": "stdio",
        "command": str(WORKSHOP_DIR / "paper_agents" / "scanpy-mcp" / ".venv" / "bin" / "python"),
        "args": [str(WORKSHOP_DIR / "paper_agents" / "scanpy-mcp" / "src" / "scanpy_mcp_stdio.py")],
        "env": {"CLUSTERING_INPUT_DIR": str(PAPER_AGENT_OUTPUTS["scanpy"] / "inputs"),
                "CLUSTERING_OUTPUT_DIR": str(PAPER_AGENT_OUTPUTS["scanpy"])},
    },
    "tissue": {
        "type": "stdio",
        "command": str(WORKSHOP_DIR / "paper_agents" / "TISSUE-mcp" / ".venv" / "bin" / "python"),
        "args": [str(WORKSHOP_DIR / "paper_agents" / "TISSUE-mcp" / "src" / "tissue_mcp_stdio.py")],
        "env": {"TISSUE_INPUT_DIR": str(PAPER_AGENT_OUTPUTS["tissue"] / "inputs"),
                "TISSUE_OUTPUT_DIR": str(PAPER_AGENT_OUTPUTS["tissue"])},
    },
}
# The Paper2Agent team's hosted copies. Good for listing tools, but they can't read files on this machine.
HOSTED_AGENTS = {
    "scanpy": {"type": "http", "url": "https://Paper2Agent-scanpy-mcp.hf.space/mcp"},
    "tissue": {"type": "http", "url": "https://Paper2Agent-tissue-mcp.hf.space/mcp"},
}

TOOL_TIMEOUT_S = 600   # single-cell steps can be slow
USE_CACHE = False      # True = load pre-run tool lists and results

# This notebook makes no model calls, so it loads no API keys. Claude Code uses your own sign-in.

In [ ]:
# Imports and helper functions. Skim the comments; you don't need to read every line.
import ast, asyncio, datetime, html, json, re, shutil, sys, time, urllib.parse
from IPython.display import display, HTML, Image

# --- Hiding secrets ------------------------------------------------------------------
# Anything that looks like a key or token is replaced before it's shown on screen or saved.
_SECRET_VALUES = set()
_SECRET_RE = re.compile(   # (?<!…) stops ordinary words like "risk-assessment" from matching
    r"(?<![A-Za-z0-9])(?:sk-[A-Za-z0-9_*.\-]{16,}|hf_[A-Za-z0-9]{20,}|gh[pousr]_[A-Za-z0-9]{20,}"
    r"|github_pat_[A-Za-z0-9_]{20,}|(?:AKIA|ASIA)[A-Z0-9]{16}\b"
    r"|eyJ[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,})")

def redact(text):
    s = "" if text is None else str(text)
    for value in _SECRET_VALUES:
        s = s.replace(value, "[hidden]")
    return _SECRET_RE.sub("[hidden]", s)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "analysis_log.json"

def _esc(x):
    return html.escape(redact(x))

def _read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return None

def table(headers, rows, title=None):
    head = "".join(f"<th style='text-align:left'>{_esc(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td style='text-align:left;vertical-align:top'>{_esc(c)}</td>" for c in r) + "</tr>" for r in rows)
    display(HTML((f"<p><b>{_esc(title)}</b></p>" if title else "") + f"<table><tr>{head}</tr>{body}</table>"))

def show_text(title, text, open_=False):
    display(HTML(f"<details{' open' if open_ else ''}><summary><b>{_esc(title)}</b></summary>"
                 f"<div style='white-space:pre-wrap;font-family:monospace;font-size:0.85em;margin:6px 0 0 12px'>{_esc(text)}</div></details>"))

def _when(s):
    try:
        return datetime.datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    except ValueError:
        return None

def _minutes(start, end):
    a, b = _when(start), _when(end)
    return f"{(b - a).total_seconds() / 60:.0f} min" if a and b else ""


# --- 1 · Touring a Paper2Agent build ----------------------------------------------
STAGES = [
    ("environment_and_selection_done",       "Clone the repo, build its environment, choose tools from its tutorials"),
    ("reference_execution_done",             "Run the tutorials to record the paper's reference results"),
    ("implementation_and_verification_done", "Write tool wrappers; fresh agents test them against the reference results"),
    ("mcp_integration_done",                 "Assemble the MCP server and call every tool"),
    ("runtime_validation_done",              "Reinstall in a clean environment and call every tool again"),
    ("delivery_done",                        "Package a ZIP with USAGE.md, unpack it elsewhere and test it again"),
]

def build_progress(project):
    """Which of Paper2Agent's six stages are finished (from its .pipeline markers)."""
    pipe = Path(project) / ".pipeline"
    if not pipe.is_dir():
        print(f"No Paper2Agent build at {project} yet (no .pipeline folder).")
        return
    rows = []
    for i, (marker, label) in enumerate(STAGES, 1):
        p = pipe / marker
        done = p.exists()
        stamp = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%d %b %H:%M") if done else ""
        rows.append(["✅" if done else "⏳", f"{i} · {label}", stamp])
    table(["", "Stage", "Finished"], rows, title=f"Build: {Path(project).name}")
    state = _read_json(Path(project) / "reports" / "agent-runs.json") or {}
    starts = [_when(r.get("started_at")) for r in state.get("runs", []) if r.get("started_at")]
    ends = [_when(r.get("finished_at")) for r in state.get("runs", []) if r.get("finished_at")]
    end = _when((state.get("completion") or {}).get("finished_at")) or (max(ends) if ends else None)
    if starts and end:
        print(f"Recorded agent work ran from {min(starts):%d %b %H:%M} to {end:%H:%M} UTC "
              f"({(end - min(starts)).total_seconds() / 3600:.1f} hours).")

def build_agents(project):
    """Which agents did what, and whether checking was independent of building."""
    state = _read_json(Path(project) / "reports" / "agent-runs.json")
    if not state:
        print("This build has no reports/agent-runs.json.")
        return
    runs = state.get("runs", [])
    icon = {"succeeded": "✅", "failed": "❌", "blocked": "⛔", "cancelled": "⏹", "running": "⏳"}
    rows = [[r.get("role"), r.get("task"), icon.get(r.get("status"), r.get("status")),
             _minutes(r.get("started_at"), r.get("finished_at")), str(r.get("agent_id", ""))[:12]] for r in runs]
    table(["Role", "Task", "Status", "Time", "Agent"], rows, title=f"{len(runs)} agent assignments")
    by_role = {}
    for r in runs:
        by_role.setdefault(r.get("role"), []).append(r)
    print("Assignments per role: " + ", ".join(f"{k} {len(v)}" for k, v in by_role.items()))
    builders = {r.get("agent_id") for r in by_role.get("implementer", [])}
    checkers = {r.get("agent_id") for r in by_role.get("verifier", [])}
    if checkers:
        print("✅ Every verifier was a different agent from every implementer." if not builders & checkers
              else f"❌ Some agents checked their own work: {sorted(builders & checkers)}")
    attempts = {}
    for r in runs:
        attempts[(r.get("role"), r.get("task"))] = attempts.get((r.get("role"), r.get("task")), 0) + 1
    retried = {k: n for k, n in attempts.items() if n > 1}
    if retried:
        print("🔁 Retried: " + ", ".join(f"{role} '{task}' ×{n}" for (role, task), n in retried.items()))

def build_selection(project):
    """What the build chose to turn into tools, and what it left out."""
    project = Path(project)
    state = _read_json(project / "reports" / "agent-runs.json") or {}
    repo = state.get("repository", {})
    if repo:
        print(f"Source: {repo.get('url')}  commit {str(repo.get('commit', 'n/a'))[:12]}\n")
    tutorials = state.get("tutorials", [])
    if tutorials:
        table(["Tutorial / example", "Becomes module", "Source file"],
              [[t.get("id"), t.get("module"), t.get("source")] for t in tutorials], title="Selected sources")
    tools = _read_json(project / "reports" / "expected-mcp-tools.json") or []
    if tools:
        print(f"Tools promised in the final server ({len(tools)}): " + ", ".join(tools))
    excl = state.get("exclusions", [])
    if excl:
        table(["Kind", "What", "Why it was left out"], [[e.get("kind"), e.get("id"), e.get("reason")] for e in excl],
              title="Deliberately left out")
    else:
        print("No recorded exclusions.")
    scan = project / "reports" / "tutorial-scanner.json"
    if scan.exists():
        show_text("The scanner's full selection report (click to expand)", scan.read_text()[:20000])

def _imports(tree):
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            names |= {a.name.split(".")[0] for a in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            names.add(node.module.split(".")[0])
    return sorted(n for n in names if n not in sys.stdlib_module_names and n not in ("fastmcp", "mcp", "typing_extensions", "pydantic"))

def _tool_defs(tree):
    """Functions decorated as MCP tools (any decorator whose name ends in 'tool')."""
    out = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            for d in node.decorator_list:
                target = d.func if isinstance(d, ast.Call) else d
                if ast.unparse(target).split(".")[-1] == "tool":
                    out.append(node)
    return out

def build_code(project):
    """The generated code: how big each tool module is, and which packages it calls into."""
    src = Path(project) / "src"
    servers = sorted(src.glob("*_mcp.py"))
    if servers:
        print("Server entry point: " + ", ".join(str(p.relative_to(project)) for p in servers))
    rows = []
    for f in sorted((src / "tools").glob("*.py")):
        try:
            tree = ast.parse(f.read_text())
        except (SyntaxError, OSError):
            continue
        tools = _tool_defs(tree)
        rows.append([f.name, len(f.read_text().splitlines()), ", ".join(t.name for t in tools) or "(helpers)",
                     ", ".join(_imports(tree))])
    if rows:
        table(["Module", "Lines", "Tools", "Calls into"], rows, title="Generated tool modules")
    else:
        print(f"No src/tools/*.py found in {project}.")

def build_tests(project):
    """How the tools were tested: acceptance cases, runtime checks and the packaged ZIP."""
    rep = Path(project) / "reports"
    cases = _read_json(rep / "mcp-acceptance-cases.json") or []
    if cases:
        rows = []
        for c in cases:
            if "error_contains" in c:
                kind, check = "🚫 should fail", f"error mentions '{c['error_contains']}'"
            else:
                kind = "✅ should work"
                bits = []
                if c.get("expected_subset"):
                    bits.append("result fields: " + ", ".join(map(str, c["expected_subset"])))
                if c.get("min_artifacts"):
                    bits.append(f"≥{c['min_artifacts']} output file(s)")
                check = "; ".join(bits)
            rows.append([c.get("name"), c.get("tool"), kind, check])
        table(["Case", "Tool", "Kind", "What is checked"], rows, title=f"{len(cases)} acceptance cases")
    for name, label in (("mcp-project-environment.json", "Build environment"),
                        ("mcp-clean-environment.json", "Fresh environment"),
                        ("delivery-validation.json", "Unpacked ZIP")):
        r = _read_json(rep / name)
        if r is None:
            print(f"⏳ {label}: no {name}")
            continue
        outcomes = r.get("cases") or []
        ok = sum(1 for c in outcomes if isinstance(c, dict) and c.get("success"))
        extra = f", {ok}/{len(outcomes)} calls passed" if outcomes else ""
        missing = f", missing tools: {r['missing']}" if r.get("missing") else ""
        print(f"{'✅' if r.get('success') else '❌'} {label}{extra}{missing}")
    tests = Path(project) / "tests" / "code"
    if tests.is_dir():
        n = len([p for p in tests.rglob("*") if p.is_file() and p.suffix in (".py", ".R")])
        print(f"🧪 {n} scientific test file(s) in tests/code/, comparing tool outputs with the reference runs.")


# --- 2 · Talking to paper agents directly ------------------------------------------
def _transport(cfg):
    from fastmcp.client.transports import StdioTransport, StreamableHttpTransport
    if cfg.get("type") == "http" or "url" in cfg:
        url = cfg["url"]
        host = urllib.parse.urlparse(url).hostname or ""
        if not (url.startswith("https://") or host in ("localhost", "127.0.0.1", "::1")):
            raise ValueError(f"refusing {url}: remote MCP servers must use HTTPS")
        return StreamableHttpTransport(url, headers=cfg.get("headers"))
    return StdioTransport(cfg["command"], cfg.get("args", []), env=cfg.get("env"), cwd=cfg.get("cwd"))

def _client(cfg, timeout):
    from fastmcp import Client
    try:
        return Client(_transport(cfg), timeout=timeout, init_timeout=180)
    except TypeError:   # older FastMCP versions have no init_timeout
        return Client(_transport(cfg), timeout=timeout)

def schema_of(tool):
    """A tool's input schema (named inputSchema in MCP SDK 1.x and input_schema in 2.x)."""
    return getattr(tool, "input_schema", None) or getattr(tool, "inputSchema", None) or {}

async def list_tools(cfg):
    async with _client(cfg, 120) as client:
        return await client.list_tools()

def _bare(tool):
    """'mcp__scanpy__run_clustering' (as Claude Code shows it) -> 'run_clustering'."""
    return tool.split("__")[-1] if tool.startswith("mcp__") else tool

async def run_tool(server, tool, arguments):
    try:   # a server that won't start, or a tool that raises, becomes an error result, not a crash
        async with _client(PAPER_AGENTS[server], TOOL_TIMEOUT_S) as client:
            try:
                r = await client.call_tool(_bare(tool), arguments, raise_on_error=False)
            except TypeError:   # older FastMCP versions have no raise_on_error, and raise on tool errors
                r = await client.call_tool(_bare(tool), arguments)
    except Exception as e:
        return {"is_error": True, "data": None, "text": f"{type(e).__name__}: {e}"}
    if isinstance(r, list):     # FastMCP before 2.10 returns the content blocks directly
        return {"is_error": False, "data": None,
                "text": "\n".join(b.text for b in r if getattr(b, "text", None))}
    text = "\n".join(b.text for b in (getattr(r, "content", None) or []) if getattr(b, "text", None))
    data = getattr(r, "structured_content", None)
    data = data if data is not None else getattr(r, "data", None)
    if isinstance(data, dict) and list(data) == ["result"]:
        data = data["result"]
    return {"is_error": bool(getattr(r, "is_error", False)), "data": data, "text": text}

def _files_in(x, found=None):
    """Every existing file path mentioned anywhere in a tool result."""
    found = [] if found is None else found
    if isinstance(x, dict):
        for v in x.values():
            _files_in(v, found)
    elif isinstance(x, (list, tuple)):
        for v in x:
            _files_in(v, found)
    elif isinstance(x, str) and x.startswith("/") and len(x) < 1000 and Path(x).is_file() and x not in found:
        found.append(x)
    return found

def show_result(res):
    if res["is_error"]:
        print(redact("❌ The tool returned an error:\n" + (res["text"] or str(res["data"]))[:3000]))
        return
    shown = res["data"] if res["data"] is not None else res["text"]
    show_text("Raw result (click to expand)", json.dumps(shown, indent=2, default=str)[:20000]
              if not isinstance(shown, str) else shown[:20000], open_=True)
    files = _files_in(res["data"]) or _files_in(re.findall(r"/[^\s'\"]+", res["text"] or ""))
    for f in [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))][:6]:
        print(f)
        display(Image(filename=f, width=480))
    others = [f for f in files if not f.lower().endswith((".png", ".jpg", ".jpeg"))]
    if others:
        print("Other output files:\n  " + "\n  ".join(others))

def _source_roots():
    roots = [Path(RECORDED_BUILD) / "src"]
    for cfg in PAPER_AGENTS.values():
        for a in cfg.get("args", []):
            if str(a).endswith(".py"):
                roots.append(Path(a).parent)
    return [r for r in dict.fromkeys(roots) if r.is_dir()]

def show_source(tool):
    """Find the function behind a tool name and show it, with the library calls it makes."""
    name = _bare(tool)
    best = None
    for root in _source_roots():
        for f in root.rglob("*.py"):
            try:
                text = f.read_text()
                tree = ast.parse(text)
            except (SyntaxError, OSError, UnicodeDecodeError):
                continue
            for node in _tool_defs(tree):
                if node.name == name or name.endswith("_" + node.name):
                    score = len(node.name)
                    if not best or score > best[0]:
                        best = (score, f, node, text, tree)
    if not best:
        print(f"Couldn't find the source of '{name}' under: " + ", ".join(map(str, _source_roots())))
        return
    _, f, node, text, tree = best
    lines = text.splitlines()[node.lineno - 1 - len(node.decorator_list): node.end_lineno]
    # Which names in this file refer to third-party packages (e.g. sc -> scanpy)?
    aliases = {}
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            for a in n.names:
                aliases[(a.asname or a.name).split(".")[0]] = a.name.split(".")[0]
        elif isinstance(n, ast.ImportFrom) and n.module and n.level == 0:
            for a in n.names:
                aliases[a.asname or a.name] = n.module.split(".")[0]
    def root(expr):
        while isinstance(expr, ast.Attribute):
            expr = expr.value
        return expr.id if isinstance(expr, ast.Name) else None
    lib_calls = sorted({f"{ast.unparse(c.func)} ({aliases[root(c.func)]})"
                        for stmt in node.body for c in ast.walk(stmt) if isinstance(c, ast.Call)
                        and root(c.func) in aliases
                        and aliases[root(c.func)] not in sys.stdlib_module_names
                        and aliases[root(c.func)] not in ("fastmcp", "mcp", "pydantic")})
    print(f"{f}  (line {node.lineno}, {len(lines)} lines)")
    print("Calls into the paper's code: " + (", ".join(lib_calls)[:1500] if lib_calls else
          "none found here. Read the code to see where the computation happens (it may be in a helper)."))
    numbered = "\n".join(f"{node.lineno - len(node.decorator_list) + i:>4}  {l}" for i, l in enumerate(lines))
    show_text(f"Source of {node.name} (click to expand)", numbered, open_=True)

def log_run(entry):
    log = _read_json(LOG_PATH) or []
    log.append(json.loads(redact(json.dumps(entry, default=str))))
    LOG_PATH.write_text(json.dumps(log, indent=2, default=str))

def show_log():
    log = _read_json(LOG_PATH) or []
    verdict = {True: "✅ matches", False: "❌ doesn't match", None: "— not judged"}
    table(["#", "Analysis", "Tool", "Error?", "Claude said", "Your check"],
          [[i, e.get("analysis_id"), f"{e.get('server')}.{e.get('tool')}", "yes" if e.get("is_error") else "no",
            (e.get("claude_said") or "")[:120], verdict.get(e.get("human_check"), str(e.get("human_check")))]
           for i, e in enumerate(log, 1)], title=f"Analysis log ({LOG_PATH})")

print(f"Outputs go to {OUTPUT_DIR}" + (" · USING PRE-RUN RESULTS" if USE_CACHE else ""))

## 1 · Walkthrough: tour a recorded build · 14:40–14:55

Paper2Agent now runs as a **skill inside Claude Code or Codex**. You give it a repository and, optionally, a tutorial to focus on. A coordinator agent then sends specialist agents through six stages, and each stage writes a marker file when its checks pass. We're touring a finished build of **Scanpy** (the "Preprocessing and clustering" tutorial), recorded before the workshop.

### 1a · The stages

In [ ]:
build_progress(RECORDED_BUILD)

### 1b · Who did the work
Each assignment is one agent with one job, recorded with timestamps. Look at the last lines: **the agents that check the tools are never the agents that wrote them.** That's the same idea as Module 1's critic, enforced as a rule.

In [ ]:
build_agents(RECORDED_BUILD)

### 1c · What became a tool, and what didn't
A tool must do something useful on *new* data, and it must call the repository's existing code. Paper2Agent may not invent algorithms to fill gaps. Anything left out is recorded with a reason.

In [ ]:
build_selection(RECORDED_BUILD)

### 1d · The generated code
Read the **Calls into** column: the science should happen in the paper's own package (here `scanpy`, `anndata`), with the wrapper only checking inputs, calling it and saving outputs. A thick wrapper is a warning sign.

In [ ]:
build_code(RECORDED_BUILD)

In [ ]:
# Show one tool's source: the first one the build promised. Change the name to look at another.
# (Without a recorded build, this shows the local Scanpy paper agent's first tool instead.)
promised = _read_json(Path(RECORDED_BUILD) / "reports" / "expected-mcp-tools.json") or ["quality_control"]
show_source(promised[0])

### 1e · Tests against the paper's results
There are three layers of checks. First, scientific tests compare each tool's output with the reference results from running the tutorial. Second, acceptance cases call every tool through MCP, including calls that *should* fail. Third, the same calls are repeated in a fresh environment and again from the unpacked ZIP.

In [ ]:
build_tests(RECORDED_BUILD)

In [ ]:
usage = Path(RECORDED_BUILD) / "USAGE.md"
show_text("USAGE.md: what someone receiving this paper agent gets (click to expand)",
          usage.read_text() if usage.exists() else "No USAGE.md in this build.")

### 💬 Ask Claude Code
Paste one of these into the Claude Code panel, replacing `<build>` with the path printed by the next cell.

1. *"`<build>` is a Paper2Agent build of Scanpy. Open the tool modules in src/tools/ and explain what one tool does, which Scanpy functions it calls and which parameters it exposes."*
2. *"Compare one tool in `<build>/src/tools/` with the tutorial notebook it came from in `<build>/notebooks/`. What did the wrapper keep, and what did it leave out?"*
3. *"Read `<build>/reports/agent-runs.json`. Which agents implemented and which verified each module, and did any verifier have to retry?"*
4. *"Look at `<build>/reports/mcp-acceptance-cases.json`. Which tool is tested least thoroughly, and what test would you add?"*

In [ ]:
print(RECORDED_BUILD)

## 2 · Hands-on: connect, ask, inspect · 14:55–15:20

### 2a · What Module 1 asked for
These are the analyses from your research strategy that need a paper agent. If you don't have a Module 1 strategy, the pre-run one is used.

In [ ]:
strategy = _read_json(STRATEGY_PATH)
if strategy is None:
    strategy = _read_json(WORKSHOP_DIR / "cache" / "module1" / "research_strategy.json")
    print("📦 No Module 1 strategy found in your outputs, so using the pre-run one.\n" if strategy else
          "No Module 1 strategy found, so using one default analysis.\n")

ANALYSES = [a for a in (strategy or {}).get("analyses", []) if str(a.get("tool")).lower() in ("scanpy", "tissue")]
if not ANALYSES:
    ANALYSES = [{"id": "A0", "tool": "scanpy",
                 "question": f"Which cell types in the tumour express {TARGET}, and how does its expression in malignant cells compare with non-malignant cells?",
                 "evidence_against": f"{TARGET} is expressed as strongly in non-malignant cells as in malignant ones"}]

table(["ID", "Paper agent", "Analysis", "Counts against the target if…"],
      [[a.get("id"), a.get("tool"), a.get("question"), a.get("evidence_against")] for a in ANALYSES])

### 2b · Meet the paper agents
This is everything Claude will know about the tools: each tool's name, description and input schema. Just as in Module 1, whatever isn't here, the agent doesn't know.

In [ ]:
INVENTORY = {}
cached_inventory = _read_json(CACHE_DIR / "inventory.json") or {}

for server in PAPER_AGENTS:
    if USE_CACHE and server in cached_inventory:
        INVENTORY[server] = cached_inventory[server]
        print(f"📦 {server}: pre-run tool list")
        continue
    try:
        tools = await asyncio.wait_for(list_tools(PAPER_AGENTS[server]), 240)
        where = "on this machine"
    except Exception as e:
        print(f"⚠️ {server}: couldn't start the local server ({type(e).__name__}: {str(e)[:150]}).")
        if server not in HOSTED_AGENTS:
            continue
        try:
            tools = await asyncio.wait_for(list_tools(HOSTED_AGENTS[server]), 120)
            where = "hosted copy (it can list tools, but can't read files on this machine)"
        except Exception as e2:
            if server in cached_inventory:
                INVENTORY[server] = cached_inventory[server]
                print(f"📦 {server}: using the pre-run tool list")
            else:
                print(redact(f"❌ {server}: the hosted copy failed too ({type(e2).__name__}: {str(e2)[:150]}). Tell a helper."))
            continue
    INVENTORY[server] = [{"name": t.name, "description": t.description or "",
                          "required": schema_of(t).get("required", []),
                          "parameters": list((schema_of(t).get("properties") or {}).keys()),
                          "schema": schema_of(t)} for t in tools]
    print(f"✅ {server}: {len(tools)} tools ({where})")

(OUTPUT_DIR / "inventory.json").write_text(json.dumps(INVENTORY, indent=2))
for server, tools in INVENTORY.items():
    table(["Tool", "Required inputs", "Description"],
          [[t["name"], ", ".join(t["required"]) or "none", (t["description"].strip().splitlines() or [""])[0][:160]] for t in tools],
          title=f"{server}: {len(tools)} tools")

In [ ]:
# The full description and input schema of one tool, exactly as Claude receives it. Change the index to see others.
listed = [(s, ts) for s, ts in INVENTORY.items() if ts]
if listed:
    server, tools = listed[0]
    t = tools[0]
    show_text(f"{server}.{t['name']}: what Claude sees",
              f"{t['description']}\n\nInput schema:\n{json.dumps(t['schema'], indent=2)}", open_=True)
else:
    print("No paper agent listed any tools, so there's nothing to show. Fix the servers in 2b first, or tell a helper.")

### 2c · Connect them to Claude Code
Claude Code's `/mcp` panel shows, approves and toggles servers, but it doesn't add them. Servers are added by listing them in a `.mcp.json` file in the folder you have open in VS Code. The next cell writes that file.

In [ ]:
mcp_file = WORKSHOP_DIR / ".mcp.json"
config = _read_json(mcp_file) or {}
if mcp_file.exists():
    shutil.copy(mcp_file, mcp_file.with_suffix(".json.bak"))
config.setdefault("mcpServers", {})
for name, cfg in PAPER_AGENTS.items():
    config["mcpServers"][name] = {**cfg, "timeout": TOOL_TIMEOUT_S * 1000}   # per-server tool timeout, in ms
mcp_file.write_text(json.dumps(config, indent=2))
print(f"✅ Wrote {mcp_file} with servers: {', '.join(config['mcpServers'])}\n")

# Codex users: the same servers in Codex's format.
codex = "\n".join(
    f'[mcp_servers.{n}]\ncommand = "{c["command"]}"\nargs = {json.dumps(c.get("args", []))}\ntool_timeout_sec = {TOOL_TIMEOUT_S}\n'
    for n, c in PAPER_AGENTS.items())
(OUTPUT_DIR / "codex_mcp.toml").write_text(codex)
print("Codex users: add this to ~/.codex/config.toml (a helper can do it for you):\n\n" + codex)

Now, in VS Code:
1. Check that the open folder is your workshop folder (**File → Open Folder…** → `~/workshop`), because that's where `.mcp.json` lives.
2. Open the Claude Code panel, start a **new conversation**, and type `/mcp`.
3. **Approve** the `scanpy` and `tissue` servers when asked. Claude Code asks once for servers listed in a project file. Both should then show as connected.
4. Ask: *"Which tools does the scanpy server give you?"* Does the answer match the table in 2b?

If a server shows as failed, run the 2b cell again. If that works but Claude Code still can't connect, start another new conversation, or reload the window (**Ctrl/Cmd+Shift+P → Developer: Reload Window**).

### 2d · Ask in plain language
Copy one prompt into Claude Code. Each prompt asks Claude to **plan before it runs** and to **use only the paper agent's tools**. Watch whether it does: a coding assistant will happily write its own Scanpy code instead, and then you're no longer running the paper's tested tools.

In [ ]:
def prompt_for(a):
    server = "tissue" if str(a.get("tool")).lower() == "tissue" else "scanpy"
    data = DATASET_PATH if server == "scanpy" else SPATIAL_DATA
    return (f"Use only the {server} MCP tools for this; don't write your own analysis code.\n"
            f"Question ({a.get('id')}): {a.get('question')}\n"
            f"Data: {data}\n"
            f"The tools save their files in {PAPER_AGENT_OUTPUTS[server]}; start every out_prefix with {a.get('id')}_.\n"
            "First list the tools you plan to call and the parameters you'll use. Then run them and report the "
            "answer in two or three sentences, with the paths of any files you made. If the tools can't answer the "
            "question, say so rather than approximating.")

for a in ANALYSES:
    print(f"──── {a.get('id')} " + "─" * 60)
    print(prompt_for(a) + "\n")

### 2e · Inspect the code that actually ran
In Claude Code, click a tool call to expand it. You'll see a name like `mcp__scanpy__<tool>` and the JSON input it sent. Copy them into `TOOL_CALL` below, then run the cell to see the function behind that tool.

In [ ]:
TOOL_CALL = {
    "analysis_id": ANALYSES[0].get("id"),
    "server": "scanpy",
    "tool": "",          # e.g. "mcp__scanpy__<tool name>", exactly as Claude Code shows it
    "arguments": {},     # paste the JSON input from the tool call
}

if TOOL_CALL["tool"]:
    show_source(TOOL_CALL["tool"])
else:
    print("Fill in TOOL_CALL from one of Claude's tool calls, then run this cell again.")

### 2f · Re-run it yourself
Call the same tool with the same arguments, directly from Python, with no language model involved. Then compare what came back with what Claude told you.

In [ ]:
result = None
if USE_CACHE:
    cached = _read_json(CACHE_DIR / "analysis_log.json") or []
    LOG_PATH.write_text(json.dumps(cached, indent=2))
    print(f"📦 Loaded {len(cached)} pre-run analysis result(s) into your log.")
elif TOOL_CALL["tool"]:
    t0 = time.time()
    result = await run_tool(TOOL_CALL["server"], TOOL_CALL["tool"], TOOL_CALL["arguments"])
    print(f"Finished in {time.time() - t0:.0f}s")
    show_result(result)
else:
    print("Fill in TOOL_CALL in 2e first.")

In [ ]:
CLAUDE_SAID = ""     # paste Claude's conclusion for this analysis, in a sentence or two
MATCHES = None       # your judgement: True, False or "partly". Does the direct re-run back up what Claude said?
NOTES = ""           # anything Module 3 should know, e.g. "clusters weren't annotated, so 'malignant' is a guess"

if result is not None:
    log_run({
        "analysis_id": TOOL_CALL["analysis_id"],
        "question": next((a.get("question") for a in ANALYSES if a.get("id") == TOOL_CALL["analysis_id"]), ""),
        "server": TOOL_CALL["server"],
        "tool": _bare(TOOL_CALL["tool"]),
        "arguments": TOOL_CALL["arguments"],
        "is_error": result["is_error"],
        "result": (json.dumps(result["data"], default=str) if result["data"] is not None else result["text"])[:4000],
        "files": _files_in(result["data"]),
        "claude_said": CLAUDE_SAID,
        "human_check": MATCHES,
        "notes": NOTES,
        "logged_at": datetime.datetime.now().isoformat(timespec="seconds"),
    })
show_log()

Repeat 2d–2f for another analysis or another tool call if you have time. Each run adds a row to the log.

### 2g · Hand-off to Module 3
The Virtual Biotech team will get the same paper agents, plus your log of what they found and how much you trust it.

In [ ]:
handoff = {
    "servers": PAPER_AGENTS,
    "hosted_servers": HOSTED_AGENTS,
    "tools": {s: [t["name"] for t in ts] for s, ts in INVENTORY.items()},
    "dataset": str(DATASET_PATH),
    "analysis_log": str(LOG_PATH),
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
}
(OUTPUT_DIR / "paper_agents.json").write_text(json.dumps(handoff, indent=2))
n = len(_read_json(LOG_PATH) or [])
print(f"✅ Saved {OUTPUT_DIR / 'paper_agents.json'}\n✅ {n} analysis result(s) in {LOG_PATH}\n")
print("That's Module 2. Next comes the short break, then the Virtual Biotech team.")

## 3 · Discussion: what makes a paper "agentifiable"? · 15:20–15:30

Paper2Agent can only build a tool if the paper's code **installs**, its tutorial **runs on real data**, and the results are **stable enough to test against**. Those are the same things a reproducibility check asks for. Here's the recorded build, scored that way:

In [ ]:
def agentifiability(project):
    project = Path(project)
    state = _read_json(project / "reports" / "agent-runs.json") or {}
    runs = state.get("runs", [])
    def count(role, status=None):
        return sum(1 for r in runs if r.get("role") == role and (status is None or r.get("status") == status))
    excl = state.get("exclusions", [])
    rep = project / "reports"
    clean = _read_json(rep / "mcp-clean-environment.json") or {}
    deliv = _read_json(rep / "delivery-validation.json") or {}
    tools = _read_json(rep / "expected-mcp-tools.json") or []
    rows = [
        ["The code installs into a fresh environment", "✅" if clean.get("success") else "❌ / not recorded"],
        ["Tutorials selected", len(state.get("tutorials", []))],
        ["Tutorials left out (see reasons in 1c)", sum(1 for e in excl if e.get("kind") == "tutorial")],
        ["Tutorial runs that needed a retry", max(0, count("executor") - len(state.get("tutorials", [])))],
        ["Tools delivered", len(tools)],
        ["Tools dropped (see reasons in 1c)", sum(1 for e in excl if e.get("kind") == "tool")],
        ["Verifier attempts (one per module, plus retries)", count("verifier")],
        ["The packaged ZIP works somewhere else", "✅" if deliv.get("success") else "❌ / not recorded"],
    ]
    table(["Reproducibility check", "Result"], rows, title=f"{project.name}, scored for reproducibility")

agentifiability(RECORDED_BUILD)

**Questions to discuss**
1. Which of Module 1's analyses had **no** paper agent? What paper or code would you need, and could it be agentified?
2. Did Claude ever write its own analysis code instead of calling the tool? Which result would you put in a methods section?
3. The tests compare each tool with reference results from **the same code**. What does that prove, and what doesn't it prove?
4. If a paper *can't* be agentified because its tutorial won't run or its results don't match, what does that tell you about the paper?

**Is your own paper agentifiable?**
- Is the code public, at a fixed version or commit?
- Does at least one tutorial or example run from start to finish on data anyone can download?
- Are the dependencies pinned, and are random seeds set?
- Does it need a GPU, licensed software, credentials or private data? (That's possible, but it must be documented.)
- Would someone else get the same numbers from the same inputs?

## ⭐ Stretch: start a build on your own repository
The build runs inside Claude Code and **takes a long time, often over an hour**, so start it now and let it finish after the session. A few things to know first:
- It runs on **your own Claude plan's usage**, not the workshop keys, and it spawns several agents, so it uses a lot.
- Keep VS Code connected while it runs. If it stops, ask Claude Code *"Resume the paper2agent build in `<folder>`"*. Paper2Agent picks up from its last finished stage.
- Choose a repository with a tutorial that runs on public data in a few minutes, on a CPU.

In [ ]:
MY_REPO  = "https://github.com/<you>/<repository>"   # your repository
MY_FOCUS = ""                                        # optional: a tutorial title or URL to focus on
MY_BUILD = WORKSHOP_DIR / "my_builds" / "My_Agent"

# Install the paper2agent skill for Claude Code (a copy of the skill folder from the Paper2Agent repository).
skill_src = PAPER2AGENT_DIR / "skills" / "paper2agent"
skill_dst = Path.home() / ".claude" / "skills" / "paper2agent"
project_skill = WORKSHOP_DIR / ".claude" / "skills" / "paper2agent"
if project_skill.exists():
    print(f"✅ The paper2agent skill is already available in this folder ({project_skill})")
elif skill_dst.exists():
    print(f"✅ The paper2agent skill is already installed at {skill_dst}")
elif skill_src.is_dir():
    shutil.copytree(skill_src, skill_dst)
    print(f"✅ Installed the paper2agent skill at {skill_dst}. Start a new Claude Code conversation so it appears.")
else:
    print(f"❌ Couldn't find the skill at {skill_src}. Tell a helper.")

focus = f" Focus on {MY_FOCUS}." if MY_FOCUS else ""
print("\nPaste this into a new Claude Code conversation:\n")
print(f"/paper2agent Convert {MY_REPO} into tested MCP tools in {MY_BUILD}.{focus}")

In [ ]:
# Check on your build at any time. Run 1b–1e on it too: build_agents(MY_BUILD), build_tests(MY_BUILD) …
build_progress(MY_BUILD)

---
### For organisers
- **The recorded build:** on your instance, before the image is made, run `/paper2agent Convert https://github.com/scverse/scanpy into tested MCP tools in ~/workshop/paper2agent_builds/Scanpy_Agent. Focus on the "Preprocessing and clustering" tutorial.` The whole project folder then travels with the image. The build runs under your own Claude sign-in, so sign out (`/logout`) afterwards; `prepare_image.py` also checks the build folder for leftover tokens.
- **The paper agents:** `python tools/setup_paper_agents.py ~/workshop` installs the Paper2Agent team's hosted Scanpy and TISSUE servers locally, unchanged, with a stdio entry point each; `PAPER_AGENTS` already points at them. Local (stdio) servers are the default: they can read the workshop dataset and open no network ports. The hosted copies can't read local files, which makes them a fallback for listing tools only. The notebook refuses any remote MCP server that isn't HTTPS. To use your own build instead, install it from its `dist/<repo>-mcp.zip` by following its `USAGE.md` and put its start-up command in `PAPER_AGENTS`.
- **Check the tools fit the question:** the Scanpy agent covers its tutorial: preprocessing and clustering. It can still compare CD276 between the dataset's annotated cell types: `annotate_cell_types` with `groupby_key="cell_type"` and `marker_genes={"B7-H3": ["CD276"]}` draws a dot plot by cell type. The full chain takes about 4 minutes on the workshop dataset, most of it in `quality_control`. TISSUE has only its own example data (mouse brain, 33 genes), so it can demonstrate the method but can't test the target: a good discussion point.
- **`/mcp` can't add servers**, so this notebook writes `~/workshop/.mcp.json` and participants approve the servers in `/mcp`. The agenda's wording ("add … with the extension's /mcp dialog") might need a small tweak.
- **Pre-run copies:** run Section 2 once with the default analyses, then copy `inventory.json`, `analysis_log.json`, `paper_agents.json` and `paper_agent_files/` (its figures and tables, plus the final annotated `.h5ad`; the intermediate `.h5ad` files add about 2.5 GB and aren't needed) from `OUTPUT_DIR` to `CACHE_DIR` before the image is made (Module 3 falls back to the cached `paper_agents.json` and `analysis_log.json`). `prepare_image.py` then empties `OUTPUT_DIR` and removes `~/workshop/.claude/settings.local.json`, where Claude Code saves your server approvals, so each participant approves the servers themselves.
- **Usage and keys:** Claude Code in 2c–2d runs on each participant's own Claude plan. This notebook loads no API keys, and the direct re-runs in 2f make no model calls. Everything shown on screen or written to the log passes through `redact()`, so anything key-shaped is hidden.